
# Post-process Chemkin DI engine solution

PyChemkin has a post-process module ``ChemkinSolutionImporter`` that can used
to import the XML solution data from selected Chemkin reactor models into Python.
This example shows how use some of objects and utilities from the
``ChemkinSolutionImporter``, the ``SolutionData``, and the ``SolutionGroup`` modules
to extract the solution variables from a transient multi-zone direct-injection (DI)
engine simulation.

The Chemkin DI engine model discretizes the liquid fuel spray into multiple
parcels (or zones) allowing for detailed tracking of fuel evaporation, mixing,
and combustion within each zone. This examples shows how to extract the
parcel-wise (zonal) information from the XML solution data. The "life-time" of
the liquid fuel (n-heptane nC\ :sub:`7`\ H\ :sub:`16`\ ) and its vapor are
compared from two parcels representing the conditions at the tip and
at the tail-end of the spray. It can be seen that as the atomized liquid fuel
droplets are injected into the spray parcel, they mix with the hot air
in the cylinder and start to evaporate leading to the formation of fuel vapor.
Subsequently, the fuel vapor decomposes and participates in the combustion
reactions within the spray parcel. The parcel at the spray tail-end exhibits
similar process but both the liquid evaporation rate and the vapor consumption
rate are higher due to the higher cylinder temperature and pressure.

In combination of other PyChemkin modules,
such as ``Chemistry`` and ``Mixture``, you can further analyze and visualize
the simulation results.

<div class="alert alert-info"><h4>Note</h4><p>The *multi-zone DI engine model* is **not** available through the PyChemkin APIs.</p></div>


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from ansys.chemkin.core.chemkin_solution_utility import ChemkinSolutionImporter
from ansys.chemkin.core.logger import logger
from ansys.chemkin.core.utilities import (
    copy_file,
    delete_files_by_extension,
)

# check working directory
current_dir = str(Path.cwd())
logger.debug("working directory: " + current_dir)

## Provide the Chemkin XML solution file
Find the data folder containing the XML solution file.



In [ ]:
try:
    script_dir_obj = Path(__file__).parent.resolve()
except NameError:
    script_dir_obj = Path(current_dir)
script_dir = str(script_dir_obj)
# use the relative path to locate the XML solution data folder
local_data_folder = script_dir_obj / ".." / "data" / "1_D_solution"
print(f"solution data folder = {str(local_data_folder)}")
# set the xml solution data file
xml_solution_file = "XMLdata_ic_engine__DI_engine.zip"
xml_solution_path = local_data_folder / xml_solution_file

## Initialize the Chemkin solution importer
Initialize the Chemkin solution importer which will store all the
raw solution data read from the XML solution file. Specify the working directory,
clean up any existing solution files there before importing new solution data.
Copy the XML solution file from the data directory to the working directory, and
provide the full path of the XML file to the importer.
The ``ChemkinSolutionImporter`` have a few options:

1. use ``get_help()`` method to get information about the Chemkin post-processor
   ``GetSolution``
2. use ``generate_preference_file()`` method to create a preference file.
   You can make changes to the preference file ``CKSolnList.txt`` to customize
   the filters and the units before running the importer. Save a copy of
   the preference file to a different location to avoid it being deleted during
   the cleanup step.
3. Once the customized preference file is ready, use the
   ``use_preference_file()`` method to apply it before reading the XML solution file.

Then use the ``import_solution()`` and the ``process_solution_data()`` methods to
import the XML solution file and to process the solution data for further processing.




In [ ]:
# Instantiate the Chemkin solution importer object
ck_solution = ChemkinSolutionImporter()
# set the working directory
ck_solution.set_working_dir(current_dir)
# clean up the working directory before copying the new solution file
delete_files_by_extension(
    targetpath=current_dir, extensions=[".ckcsv", ".txt", ".xml", ".zip"]
)
# copy the solution file to the current working directory
copy_file(str(local_data_folder), current_dir, xml_solution_file)
# set the XML solution data file for post-processing
ck_solution.set_xml_data_file(str(Path(current_dir) / xml_solution_file))
# - optional step: generate the preference file from the solution data
# the preference file name is "CKSolnList.txt".
# ck_solution.generate_preference_file(str(Path(current_dir) / xml_solution_file))
#
# - optional step: edit the preference file before generating the solution data
# pause here if you want to edit the preference file before applying it
# Remember to save the edited preference file to a different location,
# so that it won't be deleted by the cleanup step.
#
# - optional step: apply the modified preference file
# copy it to the current working directory if it is stored elsewhere
# copy_file(str(local_data_folder), current_dir, "CKSolnList.txt")
# apply the preference file
# ck_solution.use_preference_file(str(Path(current_dir) / "CKSolnList.txt"))
#
# import solution data from the XML file
ck_solution.import_solution()
# import the solution data in text format for further analysis
ck_solution.process_solution_data()

## Post-process the imported solution data
Now you can get basic information about the solution data such as
the number of solution groups, the number of solution points,
the number of solution variables, etc.
You use the methods provided by the ``ChemKinSolution`` object and/or the
``SolutionGroup`` object to manipulate the solution data for furrther analysis
and plotting.



In [ ]:
# get the list of solution group names from the imported solution data
solution_groups = ck_solution.get_solution_groups()
# print basic information about each solution group
for group in solution_groups:
    print(f"Solution Group: {group}")
    print(f"number of solution points: {ck_solution.get_number_of_points()}")
    print(f"Number of gas species: {ck_solution.get_number_of_gas_species()}")
    # print(f"variables: {ck_solution.get_variable_labels(group)}")

# print(f"species symbols: {ck_solution.get_gas_species_symbols()}")

# Instantiate a solution group data object from the first solution group in the
# solution data.
grp_idx = 0
# get the name of the first solution group
this_group = solution_groups[grp_idx]
# get the solution group object for the first solution group
this_group_obj = ck_solution.get_group_object(this_group)
# define the zonal solutions to be plotted
zone_tag = ["Zone#1", "Zone#175"]
# define the independent variable
independent_var = "Crank_rotation_angle"
# get the array of the independent variable and its unit from the solution group object
distance_series = this_group_obj.get_variable_array_with_tag(
    independent_var, zone_tag[0]
)
distance = distance_series.to_numpy(dtype=float, copy=False)
distance_unit = this_group_obj.get_variable_unit_with_tag(independent_var, zone_tag[0])
# define the solution variable to be plotted
profile_1 = {}
profile_2 = {}
for zone_name in zone_tag:
    soln_var_1 = "Liquid_fuel_mass_nc7h16"
    soln_var_2 = "nc7h16"
    soln_profile_1_series = this_group_obj.get_variable_array_with_tag(
        soln_var_1, zone_name
    )
    soln_profile_1 = soln_profile_1_series.to_numpy(dtype=float, copy=False)
    if len(soln_profile_1) <= 0:
        print(f"Error...No data available for solution variable: {soln_var_1}")
        exit()
    profile_1[zone_name] = soln_profile_1

    soln_profile_2_series = this_group_obj.get_variable_array_with_tag(
        soln_var_2, zone_name
    )
    soln_profile_2 = soln_profile_2_series.to_numpy(dtype=float, copy=False)
    if len(soln_profile_2) <= 0:
        print(f"Error...No data available for solution variable: {soln_var_2}")
        exit()
    profile_2[zone_name] = soln_profile_2
    # same unit
    soln_unit_1 = this_group_obj.get_variable_unit_with_tag(soln_var_1, zone_name)
    soln_unit_2 = this_group_obj.get_variable_unit_with_tag(soln_var_2, zone_name)

## Plot the selected solution variable
plot solution profile



In [ ]:
line_colors = ["b", "r"]
line_styles = ["-", "--"]
fig, ax1 = plt.subplots(figsize=(7.5, 5))
# Plot the primary data on the left y-axis
color1 = line_colors[0]
line_type = line_styles[0]
ax1.plot(
    distance,
    profile_1[zone_tag[0]],
    color=color1,
    linestyle=line_type,
    label=soln_var_1.replace("_", " "),
)
line_type = line_styles[1]
ax1.plot(
    distance,
    profile_1[zone_tag[1]],
    color=color1,
    linestyle=line_type,
    label=soln_var_1.replace("_", " "),
)
ax1.set_xlabel(f"{independent_var.replace('_', ' ')} {distance_unit}")
ax1.set_ylabel(
    f"{soln_var_1.replace('_', ' ')} {soln_unit_1.replace('_', ' ')}", color=color1
)
ax1.tick_params(axis="y", labelcolor=color1)
# Set x-axis limit
ax1.set_xlim(-30, 30)

# Create the secondary axes sharing the x-axis
ax2 = ax1.twinx()

color2 = line_colors[1]
line_type = line_styles[0]
ax2.plot(
    distance,
    profile_2[zone_tag[0]],
    color=color2,
    linestyle=line_type,
    label=soln_var_2.replace("_", " "),
)
line_type = line_styles[1]
ax2.plot(
    distance,
    profile_2[zone_tag[1]],
    color=color2,
    linestyle=line_type,
    label=soln_var_2.replace("_", " "),
)
ax2.set_xlabel(f"{independent_var.replace('_', ' ')} {distance_unit}")
ax2.set_ylabel(
    f"{soln_var_2.replace('_', ' ')} {soln_unit_2.replace('_', ' ')}", color=color2
)
ax2.tick_params(axis="y", labelcolor=color2)
# Set x-axis limit
ax2.set_xlim(-30, 30)
#
plt.legend(
    [
        f"{zone_tag[0].replace('#', ' ')} : spray front",
        f"{zone_tag[1].replace('#', ' ')} : spray end",
    ],
    loc="best",
)
# plot results
plt.show()
# plt.savefig("plot_post_process_DI_engine.png", bbox_inches="tight")